# Backtesting

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
import plotly.express as px

#### 1. Load predictions & price Data

In [0]:
preds_df = (
    spark.table(tables["cnn_lstm_predictions"])
    .select("symbol", "date", "predicted")
    .filter((F.col("date") >= test_start) & (F.col("date") <= test_end))
    .toPandas()
)

prices_df = (
    spark.table(tables["cleaned_ohlcv"])
    .select("symbol", "date", "open", "close")
    .filter(F.col("date") >= test_start)
    .toPandas()
)

preds_df["date"] = pd.to_datetime(preds_df["date"])
prices_df["date"] = pd.to_datetime(prices_df["date"])
prices_df = prices_df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"Predictions: {len(preds_df):,}  |  Daily price rows: {len(prices_df):,}")

#### 2. Compute forward returns

In [0]:
pg = prices_df.groupby("symbol")
prices_df["entry_price"] = pg["open"].shift(-1)
for h in event_study_list:
    prices_df[f"ret_{h}"] = pg["close"].shift(-h) / prices_df["entry_price"] - 1.0

df = preds_df.merge(prices_df, on=["symbol", "date"], how="inner")
df["phase_name"] = df["predicted"].map(phase_names)

print(f"After merge: {len(df):,} rows  |  Symbols: {df['symbol'].nunique()}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Forward-return horizons: {event_study_list}")

#### 3. Event study 

In [0]:
rows = []
for phase_id in range(number_of_phases):
    sub = df[df["predicted"] == phase_id]
    row = {"phase": phase_names[phase_id], "n_signals": len(sub)}
    for h in event_study_list:
        r = sub[f"ret_{h}"].dropna()
        n = len(r)
        mean = r.mean()
        std = r.std()
        tstat = mean / (std / np.sqrt(n)) if n > 1 and std > 0 else np.nan
        row[f"mean_{h}d_pct"] = round(mean * 100, 2)
        row[f"tstat_{h}d"] = round(tstat, 2)
        row[f"win_{h}d_pct"] = round((r > 0).mean() * 100, 1)
    rows.append(row)

event_study = pd.DataFrame(rows)

print("Event Study Result")
print(event_study.to_string(index=False))

#### 4. Visualize

In [0]:
plot_df = event_study.melt(
    id_vars=["phase", "n_signals"],
    value_vars=[f"mean_{h}d_pct" for h in event_study_list],
    var_name="horizon",
    value_name="mean_return_pct",
)
plot_df["horizon"] = plot_df["horizon"].str.extract(r"(\d+)d").astype(int)

fig = px.bar(
    plot_df,
    x="phase",
    y="mean_return_pct",
    color="horizon",
    barmode="group",
    title="Event Study",
    labels={"mean_return_pct": "Mean forward return (%)", "phase": "Predicted phase", "horizon": "Horizon (days)"},
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(height=480)
fig.show()


#### 5. Save


In [0]:
(spark.createDataFrame(event_study).write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tables["event_study_results"]))

print(f"Saved: {tables['event_study_results']} ({len(event_study)} rows)")